
# v4 Protocol Viability Simulation

This notebook simulates a **v4 protocol design** focused on **joint viability** for LP hedgers and RT insurers (not benchmark superiority vs external derivatives).

Core v4 pricing design:
- Fair-value baseline (kept from v3 style corridor expected payout).
- Volatility markup decomposition: realized-vol component + IV/RV component.
- AMM-like multiplicative markup from demand/supply imbalance using a quadratic bonding curve.

Contract flow simplification:
- LP locks position + pays premium, receives NFT-like hedge contract (modeled as a position record).
- At expiry, position is closed; LP gets position outcome + hedge settlement.
- RT deposits insurer capital, underwrites contracts, receives premium upfront + yield share.
- RT pays compensation at settlement; idle RT capital can earn lending yield.
- Leveraged LP position scenarios are included.


In [1]:

import os
import math
import time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss
from datetime import datetime, timezone
import requests

plt.style.use('seaborn-v0_8-whitegrid')
rng = np.random.default_rng(42)

DATA_DIR = 'data'
os.makedirs(DATA_DIR, exist_ok=True)

T_WEEK = 7 / 365
N_LIQ = 10_000
WIDTH = 0.075  # +/-7.5%
BARRIER_BPS = 750  # full corridor to lower bound
DAILY_FEE = 0.0055

print('Environment initialized.')


Environment initialized.


In [2]:

# --- Data loader (Birdeye with robust fallback) ---
BIRDEYE_KEY = 'ed577a4a6a4f480fa659b4f18673e4b1'
SOL_MINT = 'So11111111111111111111111111111111111111112'


def fetch_birdeye_ohlcv(days_back=730, chunk_days=90):
    url = 'https://public-api.birdeye.so/defi/ohlcv'
    headers = {'X-API-KEY': BIRDEYE_KEY, 'Accept': 'application/json'}
    now = int(time.time())
    t = now - days_back * 86400
    all_items = []
    while t < now:
        t2 = min(t + chunk_days * 86400, now)
        params = {'address': SOL_MINT, 'type': '1D', 'time_from': t, 'time_to': t2}
        try:
            js = requests.get(url, headers=headers, params=params, timeout=20).json()
            all_items.extend(js.get('data', {}).get('items', []))
        except Exception:
            pass
        t = t2
        time.sleep(0.15)

    dedup = {}
    for it in all_items:
        ts = it.get('unixTime', it.get('time', 0))
        dedup[ts] = float(it.get('c', 0.0))
    arr = sorted([(k, v) for k, v in dedup.items() if v > 0], key=lambda x: x[0])
    if not arr:
        return np.array([]), np.array([])
    ts = np.array([x[0] for x in arr], dtype=int)
    px = np.array([x[1] for x in arr], dtype=float)
    return ts, px


ts, closes = fetch_birdeye_ohlcv()
if len(closes) == 0:
    n = 730
    mu = 0.0
    sig = 0.70 / np.sqrt(365)
    z = rng.standard_normal(n)
    closes = 100.0 * np.exp(np.cumsum((mu - 0.5 * sig * sig) + sig * z))
    ts = np.arange(n)

log_ret = np.diff(np.log(closes))
vol_full = float(np.std(log_ret, ddof=1) * np.sqrt(365))

print(f'Data points: {len(closes)} | annualized vol: {vol_full:.2%}')


Data points: 730 | annualized vol: 81.76%


In [3]:

# --- v3-style fair value baseline math ---
def cl_value_vec(S, L, p_l, p_u):
    S = np.atleast_1d(np.asarray(S, float))
    spl, spu = np.sqrt(p_l), np.sqrt(p_u)
    sp = np.sqrt(np.clip(S, 1e-10, None))
    bl = S <= p_l
    ab = S >= p_u
    a = np.where(bl, L * (spu - spl) / (spl * spu), np.where(ab, 0.0, L * (spu - sp) / (sp * spu)))
    b = np.where(bl, 0.0, np.where(ab, L * (spu - spl), L * (sp - spl)))
    return a * S + b


def corridor_payoff_vec(S_T, S0, B, Cap, L, p_l, p_u):
    S_T = np.atleast_1d(np.asarray(S_T, float))
    V0 = cl_value_vec(S0, L, p_l, p_u)
    V_eff = cl_value_vec(np.maximum(S_T, B), L, p_l, p_u)
    return np.minimum(Cap, np.maximum(0.0, np.where(S_T >= S0, 0.0, V0 - V_eff)))


def fv_quadrature(S0, sigma, B, Cap, L, p_l, p_u, T=T_WEEK):
    nodes, weights = hermgauss(96)
    S_T = S0 * np.exp(-sigma**2 / 2 * T + sigma * np.sqrt(T) * nodes * np.sqrt(2))
    pay = corridor_payoff_vec(S_T, S0, B, Cap, L, p_l, p_u)
    return max(0.0, float(np.sum(weights * pay) / np.sqrt(np.pi)))


def make_position(S0, leverage=1.0, width=WIDTH, barrier_bps=BARRIER_BPS):
    p_l = S0 * (1 - width)
    p_u = S0 * (1 + width)
    barrier = max(S0 * (1 - barrier_bps / 10_000), p_l)
    V0 = float(cl_value_vec(np.array([S0]), N_LIQ * leverage, p_l, p_u)[0])
    Vb = float(cl_value_vec(np.array([barrier]), N_LIQ * leverage, p_l, p_u)[0])
    nat_cap = max(0.0, V0 - Vb)
    return {'p_l': p_l, 'p_u': p_u, 'barrier': barrier, 'V0': V0, 'nat_cap': nat_cap, 'L': N_LIQ * leverage}

print('Pricing baseline functions ready.')


Pricing baseline functions ready.


In [4]:

# --- v4 premium model ---

# Explicit pricing bounds (floors/caps)
VOL_MARKUP_FLOOR = 0.70
VOL_MARKUP_CAP = 1.80
AMM_MULT_FLOOR = 0.75
AMM_MULT_CAP = 1.60


def vol_markup(rv30, rv7, ivrv,
               alpha_rv=0.25,
               alpha_ivrv=0.35,
               rv_ref=0.60,
               floor=VOL_MARKUP_FLOOR,
               cap=VOL_MARKUP_CAP):
    rv_term = alpha_rv * ((rv30 / max(rv_ref, 1e-9)) - 1.0)
    ivrv_term = alpha_ivrv * (ivrv - 1.0)
    return float(np.clip(1.0 + rv_term + ivrv_term, floor, cap))


def amm_multiplier(demand_notional, supply_notional,
                   k_lin=0.20,
                   k_quad=0.25,
                   floor=AMM_MULT_FLOOR,
                   cap=AMM_MULT_CAP):
    if supply_notional <= 0:
        return cap
    imbalance = demand_notional / supply_notional - 1.0
    mult = 1.0 + k_lin * imbalance + k_quad * np.sign(imbalance) * (imbalance ** 2)
    return float(np.clip(mult, floor, cap))


def premium_v4(fair_value, rv30, rv7, ivrv, demand_notional, supply_notional,
               alpha_rv=0.25, alpha_ivrv=0.35,
               k_lin=0.20, k_quad=0.25,
               vol_floor=VOL_MARKUP_FLOOR, vol_cap=VOL_MARKUP_CAP,
               amm_floor=AMM_MULT_FLOOR, amm_cap=AMM_MULT_CAP):
    m_vol = vol_markup(rv30, rv7, ivrv, alpha_rv=alpha_rv, alpha_ivrv=alpha_ivrv, floor=vol_floor, cap=vol_cap)
    m_amm = amm_multiplier(demand_notional, supply_notional, k_lin=k_lin, k_quad=k_quad, floor=amm_floor, cap=amm_cap)
    premium = fair_value * m_vol * m_amm
    return premium, m_vol, m_amm

print('v4 premium model ready.')


v4 premium model ready.


In [5]:

# --- Weekly market simulator (LP demand vs RT supply) ---

def rolling_vol(prices, window):
    lr = np.diff(np.log(prices))
    out = np.full(len(prices), np.nan)
    for i in range(window, len(lr) + 1):
        out[i] = float(np.std(lr[i-window:i], ddof=1) * np.sqrt(365))
    return out

vol_7 = rolling_vol(closes, 7)
vol_30 = rolling_vol(closes, 30)


def simulate_v4(
    closes,
    vol_7,
    vol_30,
    leverage_levels=(1.0, 1.5, 2.0),
    leverage_probs=(0.55, 0.30, 0.15),
    rt_initial_capital=2_000_000.0,
    max_utilization=0.70,
    yield_share_to_rt=0.10,
    lend_rate_weekly=0.0015,
    alpha_rv=0.25,
    alpha_ivrv=0.35,
    k_lin=0.20,
    k_quad=0.25,
    demand_intensity=4.0,
    vol_floor=VOL_MARKUP_FLOOR,
    vol_cap=VOL_MARKUP_CAP,
    amm_floor=AMM_MULT_FLOOR,
    amm_cap=AMM_MULT_CAP,
):
    rows = []
    rt_capital = rt_initial_capital

    for si in range(35, len(closes)-7, 7):
        ei = si + 7
        S_s = float(closes[si])
        S_e = float(closes[ei])
        path = closes[si:ei+1]

        rv30 = float(vol_30[si]) if np.isfinite(vol_30[si]) else 0.65
        rv7 = float(vol_7[si]) if np.isfinite(vol_7[si]) else rv30
        ivrv = float(np.clip(0.95 + 0.45 * (rv7 / max(rv30, 1e-9)), 0.90, 1.45))

        # Demand stochastically rises in high short-term vol.
        lam = demand_intensity * np.clip(rv7 / max(rv30, 1e-9), 0.6, 1.8)
        n_demand = int(rng.poisson(lam=max(lam, 0.2)))
        if n_demand <= 0:
            rt_capital *= (1.0 + lend_rate_weekly)
            rows.append({'week': len(rows), 'contracts': 0, 'lp_mean_ret': 0.0, 'rt_ret': lend_rate_weekly,
                         'rt_capital': rt_capital, 'utilization': 0.0,
                         'premium_mean': 0.0, 'fair_mean': 0.0, 'm_vol': 1.0, 'm_amm': 1.0})
            continue

        # Build requested contracts
        req = []
        for _ in range(n_demand):
            lev = float(rng.choice(leverage_levels, p=leverage_probs))
            pos = make_position(S_s, leverage=lev)
            fv = fv_quadrature(S_s, rv30, pos['barrier'], pos['nat_cap'], pos['L'], pos['p_l'], pos['p_u'])
            req.append((lev, pos, fv))

        demand_notional = float(sum(x[1]['nat_cap'] for x in req))
        supply_notional = float(max(0.0, rt_capital * max_utilization))

        # Price with market-clearing AMM multiplier.
        sample_fv = float(np.mean([x[2] for x in req]))
        sample_premium, m_vol, m_amm = premium_v4(sample_fv, rv30, rv7, ivrv, demand_notional, supply_notional,
                                                  alpha_rv=alpha_rv, alpha_ivrv=alpha_ivrv,
                                                  k_lin=k_lin, k_quad=k_quad,
                                                  vol_floor=vol_floor, vol_cap=vol_cap,
                                                  amm_floor=amm_floor, amm_cap=amm_cap)

        # Capacity-constrained underwriting.
        covered = []
        used_cap = 0.0
        for lev, pos, fv in req:
            reserve_needed = pos['nat_cap']
            if used_cap + reserve_needed <= supply_notional:
                covered.append((lev, pos, fv))
                used_cap += reserve_needed

        # Economics for covered contracts.
        lp_returns = []
        rt_premium_income = 0.0
        rt_claims = 0.0
        rt_yield_share = 0.0

        for lev, pos, fv in covered:
            V0 = pos['V0']
            V_end = float(cl_value_vec(np.array([S_e]), pos['L'], pos['p_l'], pos['p_u'])[0])
            IL = V_end - V0
            in_rng = float(np.mean((path[1:] >= pos['p_l']) & (path[1:] <= pos['p_u'])))
            fees = V0 * DAILY_FEE * 7 * in_rng

            prem, _, _ = premium_v4(fv, rv30, rv7, ivrv, demand_notional, supply_notional,
                                    alpha_rv=alpha_rv, alpha_ivrv=alpha_ivrv,
                                    k_lin=k_lin, k_quad=k_quad,
                                    vol_floor=vol_floor, vol_cap=vol_cap,
                                    amm_floor=amm_floor, amm_cap=amm_cap)

            # Full-protection payout: LP does not bear IL+depreciation loss (within protected corridor cap).
            protected_loss = float(min(pos['nat_cap'], max(0.0, V0 - V_end)))
            payout = protected_loss

            # LP receives position result + residual yield share - premium + payout.
            lp_ret = (IL + fees * (1.0 - yield_share_to_rt) + payout - prem) / max(V0, 1e-9)
            lp_returns.append(lp_ret)

            rt_premium_income += prem
            rt_claims += payout
            rt_yield_share += fees * yield_share_to_rt

        idle_cap = max(0.0, rt_capital - used_cap)
        lending_income = idle_cap * lend_rate_weekly
        rt_profit = rt_premium_income + rt_yield_share + lending_income - rt_claims
        rt_ret = rt_profit / max(rt_capital, 1e-9)
        rt_capital = max(1.0, rt_capital + rt_profit)

        rows.append({
            'week': len(rows),
            'contracts': len(covered),
            'contracts_requested': n_demand,
            'lp_mean_ret': float(np.mean(lp_returns)) if lp_returns else 0.0,
            'lp_med_ret': float(np.median(lp_returns)) if lp_returns else 0.0,
            'rt_ret': float(rt_ret),
            'rt_capital': float(rt_capital),
            'utilization': float(used_cap / max(rt_capital, 1e-9)),
            'premium_mean': float(sample_premium),
            'fair_mean': float(sample_fv),
            'm_vol': float(m_vol),
            'm_amm': float(m_amm),
            'demand_notional': float(demand_notional),
            'supply_notional': float(supply_notional),
            'ivrv': float(ivrv),
            'rv30': float(rv30),
            'rv7': float(rv7),
        })

    return pd.DataFrame(rows)

print('Simulator ready.')


Simulator ready.


In [6]:

# --- Scenario sweep: viability focus (LP + RT) ---
SCENARIOS = [
    {'name': 'balanced_base', 'yield_share_to_rt': 0.10, 'k_lin': 0.20, 'k_quad': 0.25, 'alpha_rv': 0.25, 'alpha_ivrv': 0.35, 'demand_intensity': 4.0},
    {'name': 'rt_friendly',   'yield_share_to_rt': 0.15, 'k_lin': 0.25, 'k_quad': 0.30, 'alpha_rv': 0.30, 'alpha_ivrv': 0.40, 'demand_intensity': 4.0},
    {'name': 'lp_friendly',   'yield_share_to_rt': 0.06, 'k_lin': 0.12, 'k_quad': 0.18, 'alpha_rv': 0.18, 'alpha_ivrv': 0.28, 'demand_intensity': 4.5},
    {'name': 'high_demand',   'yield_share_to_rt': 0.10, 'k_lin': 0.22, 'k_quad': 0.32, 'alpha_rv': 0.25, 'alpha_ivrv': 0.35, 'demand_intensity': 6.0},
    {'name': 'low_demand',    'yield_share_to_rt': 0.10, 'k_lin': 0.18, 'k_quad': 0.20, 'alpha_rv': 0.25, 'alpha_ivrv': 0.35, 'demand_intensity': 2.6},
]

summary = []
paths = {}

for sc in SCENARIOS:
    df_sc = simulate_v4(
        closes, vol_7, vol_30,
        yield_share_to_rt=sc['yield_share_to_rt'],
        k_lin=sc['k_lin'],
        k_quad=sc['k_quad'],
        alpha_rv=sc['alpha_rv'],
        alpha_ivrv=sc['alpha_ivrv'],
        demand_intensity=sc['demand_intensity'],
    )
    paths[sc['name']] = df_sc

    lp = df_sc['lp_mean_ret'].to_numpy(dtype=float)
    rt = df_sc['rt_ret'].to_numpy(dtype=float)
    util = df_sc['utilization'].to_numpy(dtype=float)

    lp_med = float(np.median(lp))
    lp_mean = float(np.mean(lp))
    lp_cvar5 = float(np.mean(np.sort(lp)[:max(1, int(0.05 * len(lp)))]))
    rt_mean = float(np.mean(rt))
    rt_med = float(np.median(rt))
    rt_cap_growth = float(df_sc['rt_capital'].iloc[-1] / max(df_sc['rt_capital'].iloc[0], 1e-9) - 1.0)

    # Strict joint viability target: both agents have positive mean returns.
    viable = (lp_mean > 0.0 and rt_mean > 0.0)

    summary.append({
        'scenario': sc['name'],
        'lp_med_%': lp_med * 100,
        'lp_mean_%': lp_mean * 100,
        'lp_cvar5_%': lp_cvar5 * 100,
        'rt_med_%': rt_med * 100,
        'rt_mean_%': rt_mean * 100,
        'rt_cap_growth_%': rt_cap_growth * 100,
        'util_mean_%': float(np.mean(util) * 100),
        'contracts_mean': float(df_sc['contracts'].mean()),
        'premium_over_fv_mean': float(np.mean(df_sc['premium_mean'] / np.maximum(df_sc['fair_mean'], 1e-9))),
        'viable': 'YES' if viable else 'NO',
    })

sum_df = pd.DataFrame(summary).sort_values(['viable', 'lp_mean_%', 'rt_mean_%'], ascending=[False, False, False])
print('Current instantiated pricing bounds:')
print(f'  vol markup floor/cap = [{VOL_MARKUP_FLOOR:.2f}, {VOL_MARKUP_CAP:.2f}]')
print(f'  AMM multiplier floor/cap = [{AMM_MULT_FLOOR:.2f}, {AMM_MULT_CAP:.2f}]')
print(sum_df.to_string(index=False, justify='center', float_format=lambda x: f'{x:,.3f}'))


Current instantiated pricing bounds:
  vol markup floor/cap = [0.70, 1.80]
  AMM multiplier floor/cap = [0.75, 1.60]
   scenario    lp_med_%  lp_mean_%  lp_cvar5_%  rt_med_%  rt_mean_%  rt_cap_growth_%  util_mean_%  contracts_mean  premium_over_fv_mean viable
   low_demand   1.062      0.029     -12.329     0.155     0.150         15.976         0.077          2.475              0.785          YES  
  lp_friendly   1.286     -0.117     -15.349     0.161     0.144         15.198         0.132          4.323              0.851           NO  
  high_demand   1.143     -0.301     -15.471     0.164     0.145         15.444         0.180          5.788              0.878           NO  
balanced_base   1.138     -0.314     -15.471     0.158     0.147         15.653         0.127          4.040              0.874           NO  
  rt_friendly   0.921     -0.446     -15.585     0.159     0.150         15.921         0.116          3.677              0.901           NO  


In [7]:

# --- Visual diagnostics for top viable scenario ---
viable_rows = sum_df[sum_df['viable'] == 'YES']
pick = viable_rows.iloc[0]['scenario'] if len(viable_rows) else sum_df.iloc[0]['scenario']
sel = paths[pick]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0,0].plot(sel['lp_mean_ret'].to_numpy() * 100, color='seagreen', lw=1.3)
axes[0,0].axhline(0, color='black', ls='--', lw=0.8)
axes[0,0].set_title(f'LP weekly return (%) - {pick}')

axes[0,1].plot(sel['rt_ret'].to_numpy() * 100, color='firebrick', lw=1.3)
axes[0,1].axhline(0, color='black', ls='--', lw=0.8)
axes[0,1].set_title('RT weekly return (%)')

axes[1,0].plot(sel['premium_mean'].to_numpy(), label='premium', color='royalblue')
axes[1,0].plot(sel['fair_mean'].to_numpy(), label='fair value', color='gray', alpha=0.8)
axes[1,0].set_title('Premium vs Fair Value (USD)')
axes[1,0].legend(frameon=False)

axes[1,1].plot(sel['m_vol'].to_numpy(), label='vol markup', color='darkorange')
axes[1,1].plot(sel['m_amm'].to_numpy(), label='amm multiplier', color='purple')
axes[1,1].set_title('Markup Components')
axes[1,1].legend(frameon=False)

for ax in axes.flat:
    ax.set_xlabel('Week')

plt.tight_layout()
out = os.path.join(DATA_DIR, 'v4_viability_diagnostics.png')
fig.savefig(out, dpi=150)
print(f'Selected scenario: {pick}')
print(f'Saved: {out}')


Selected scenario: low_demand
Saved: data/v4_viability_diagnostics.png


In [8]:

# --- Export scenario summary ---
out_csv = os.path.join(DATA_DIR, 'v4_viability_summary.csv')
sum_df.to_csv(out_csv, index=False)
print('Saved:', out_csv)


Saved: data/v4_viability_summary.csv


## Viability Region Search (LP mean > 0 and RT mean > 0)

This section scans pricing-control parameters and maps the region where both LP hedgers and RT insurers have positive simulated mean returns.

In [9]:
# Grid search for viability region.
region_rows = []

# Keep model simple: vary a compact set of economically interpretable controls.
for alpha_rv in [0.10, 0.18, 0.25, 0.32]:
    for alpha_ivrv in [0.20, 0.30, 0.40, 0.50]:
        for k_lin in [0.10, 0.18, 0.26]:
            for k_quad in [0.10, 0.22, 0.34]:
                for yshare in [0.05, 0.08, 0.12, 0.15]:
                    for util in [0.60, 0.70, 0.80]:
                        df_g = simulate_v4(
                            closes, vol_7, vol_30,
                            alpha_rv=alpha_rv,
                            alpha_ivrv=alpha_ivrv,
                            k_lin=k_lin,
                            k_quad=k_quad,
                            yield_share_to_rt=yshare,
                            max_utilization=util,
                            demand_intensity=4.0,
                            vol_floor=0.75,
                            vol_cap=1.55,
                            amm_floor=0.80,
                            amm_cap=1.45,
                        )
                        lp_mean = float(df_g['lp_mean_ret'].mean())
                        rt_mean = float(df_g['rt_ret'].mean())
                        lp_med = float(df_g['lp_med_ret'].median())
                        rt_cap_growth = float(df_g['rt_capital'].iloc[-1] / max(df_g['rt_capital'].iloc[0], 1e-9) - 1.0)

                        viable = (lp_mean > 0 and rt_mean > 0)
                        region_rows.append({
                            'alpha_rv': alpha_rv,
                            'alpha_ivrv': alpha_ivrv,
                            'k_lin': k_lin,
                            'k_quad': k_quad,
                            'yield_share_to_rt': yshare,
                            'max_utilization': util,
                            'lp_mean_%': lp_mean * 100,
                            'lp_med_%': lp_med * 100,
                            'rt_mean_%': rt_mean * 100,
                            'rt_cap_growth_%': rt_cap_growth * 100,
                            'premium_over_fv_mean': float(np.mean(df_g['premium_mean'] / np.maximum(df_g['fair_mean'], 1e-9))),
                            'viable': viable,
                        })

region_df = pd.DataFrame(region_rows)
viable_df = region_df[region_df['viable']].copy()
print(f'Total grid points: {len(region_df)}')
print(f'Viable points (LP mean > 0 and RT mean > 0): {len(viable_df)}')

if len(viable_df) > 0:
    top = viable_df.sort_values(['lp_mean_%', 'rt_mean_%'], ascending=False).head(15)
    print('\nTop viable parameter points:')
    print(top.to_string(index=False, float_format=lambda x: f'{x:,.3f}'))
else:
    print('No viable region found under current ranges. Widen ranges or relax constraints.')

Total grid points: 1728
Viable points (LP mean > 0 and RT mean > 0): 26

Top viable parameter points:
 alpha_rv  alpha_ivrv  k_lin  k_quad  yield_share_to_rt  max_utilization  lp_mean_%  lp_med_%  rt_mean_%  rt_cap_growth_%  premium_over_fv_mean  viable
    0.100       0.200  0.260   0.100              0.050            0.600      0.235     1.511      0.146           15.399                 0.864    True
    0.100       0.200  0.260   0.220              0.050            0.700      0.219     1.506      0.145           15.350                 0.838    True
    0.100       0.500  0.180   0.100              0.050            0.800      0.209     1.425      0.150           16.077                 0.907    True
    0.180       0.200  0.180   0.100              0.080            0.700      0.207     1.286      0.151           16.078                 0.867    True
    0.180       0.300  0.100   0.340              0.050            0.800      0.204     1.360      0.146           15.655                 

In [10]:
# Visualize viability region in 2D slices.
if len(region_df) == 0:
    print('Run grid first.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Slice 1: alpha_ivrv vs yield_share_to_rt, viability ratio.
    p1 = region_df.pivot_table(
        index='alpha_ivrv', columns='yield_share_to_rt', values='viable', aggfunc='mean'
    )
    im1 = axes[0].imshow(p1.values, origin='lower', aspect='auto', cmap='viridis', vmin=0, vmax=1)
    axes[0].set_title('Viability ratio by (alpha_ivrv, yield_share)')
    axes[0].set_xticks(range(len(p1.columns)))
    axes[0].set_xticklabels([f'{x:.2f}' for x in p1.columns], rotation=45)
    axes[0].set_yticks(range(len(p1.index)))
    axes[0].set_yticklabels([f'{y:.2f}' for y in p1.index])
    axes[0].set_xlabel('yield_share_to_rt')
    axes[0].set_ylabel('alpha_ivrv')
    fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

    # Slice 2: k_lin vs k_quad, viability ratio.
    p2 = region_df.pivot_table(
        index='k_quad', columns='k_lin', values='viable', aggfunc='mean'
    )
    im2 = axes[1].imshow(p2.values, origin='lower', aspect='auto', cmap='plasma', vmin=0, vmax=1)
    axes[1].set_title('Viability ratio by (k_lin, k_quad)')
    axes[1].set_xticks(range(len(p2.columns)))
    axes[1].set_xticklabels([f'{x:.2f}' for x in p2.columns])
    axes[1].set_yticks(range(len(p2.index)))
    axes[1].set_yticklabels([f'{y:.2f}' for y in p2.index])
    axes[1].set_xlabel('k_lin')
    axes[1].set_ylabel('k_quad')
    fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    out = os.path.join(DATA_DIR, 'v4_viability_region.png')
    fig.savefig(out, dpi=150)
    print('Saved:', out)

    # Export region table
    out_csv = os.path.join(DATA_DIR, 'v4_viability_region_points.csv')
    region_df.to_csv(out_csv, index=False)
    print('Saved:', out_csv)

Saved: data/v4_viability_region.png
Saved: data/v4_viability_region_points.csv
